# 强化学习社招面试准备清单

下面按**算法原理 → 工程实践 → 系统设计**三个维度整理，适合强化学习方向社招面试准备。

| 维度 | 重点 | 目标 |
|---|---|---|
| 算法原理 | RL 基础、价值/策略方法、PPO/DPO/GRPO、RLHF、Reward Model | 能讲清楚原理、公式、优缺点 |
| 工程实践 | 分布式训练、训练稳定性、Reward 建模、数据管理、调试 | 能说明真实训练中的坑和解决方案 |
| 系统设计 | 大模型后训练链路、Agentic RL、Reward System、评估体系 | 能设计完整系统，并说明扩展性和风险 |

## 算法原理：必须扎实
社招强化学习面试通常不会只问“PPO 是什么”，而是会追问：为什么这样设计、哪里会崩、怎么改、和 DPO/GRPO 有什么区别。

### RL 基础

需要熟练掌握：
#### MDP 建模
  - 状态、动作、奖励、转移概率、折扣因子
  - 为什么需要折扣因子：有限回报、数值稳定、长期偏好建模
  
#### 价值函数
  - 状态价值： V(s)
  - 动作价值： Q(s,a)
  - 优势函数： A(s,a)=Q(s,a)−V(s)
#### Bellman 方程
  - 价值函数的递归关系
  - TD target：
    - $r_t + γV(s_{t+1})$
#### MSE 在价值回归中的作用：
  $L(θ)=E[(y_{target}−V_θ(s))^2]$
#### 策略梯度
  - 目标函数：J(θ)=E_πθ[R]
#### 策略梯度形式：

  $∇_θJ(θ)=E[∇_θlogπθ(a∣s)A(s,a)]$
  
### 经典算法
重点掌握：

| 算法 | 面试重点 |
|---|---|
| DQN | Experience Replay、Target Network、Double DQN |
| REINFORCE | 高方差、baseline、为什么难训练 |
| Actor-Critic | Actor 更新策略，Critic 估计价值 |
| A2C/A3C | 优势估计、并行采样 |
| PPO | clip、KL 约束、GAE、训练稳定性 |
| SAC | 最大熵 RL、自动温度调节 |
| DPO | 偏好学习、隐式 reward、避免显式 RM 训练 |
| GRPO/ORPO | 大模型对齐中的变体思路 |

### PPO 必须深挖
PPO 是高频考点，建议能讲清楚：
#### 目标函数
ratio：

#### clipped surrogate objective：

#### 为什么要 clip
  - 防止策略更新过大
  - 避免一步更新偏离旧策略太远
  - 提高训练稳定性
#### GAE
  - 平衡偏差和方差
  - λ 越大，越接近蒙特卡洛，方差高、偏差低
  - λ 越小，越接近 TD，偏差高、方差低
#### 常见失败模式
  - reward hacking
  - KL 散度爆炸
  - value loss 不稳定
  - advantage 尺度异常
  - 策略过早坍缩
  
### RLHF / DPO / GRPO
大模型后训练方向必须掌握。
#### RLHF 链路
  1. SFT 预训练后模型
  2. 收集人类偏好数据
  3. 训练 Reward Model
  4. 用 PPO 优化策略
  5. 可选：加入 KL 约束，防止偏离 SFT 模型太远
#### Reward Model 损失
- Bradley-Terry 模型：

$P(y_w ≻ yl)=σ(r(y_w)−r(yl))$
- 优化目标：
− $−logσ(r(y_w)−r(yl))$

#### DPO
  - 不显式训练 Reward Model
  - 直接优化偏好数据
  - 本质是把 reward 用 policy 表示出来
  - 优点：训练更稳定
  - 缺点：依赖偏好数据质量，离线约束较强
#### GRPO / ORPO 类方法
  - 重点理解它们相对 PPO/DPO 的改进动机
  - 例如：减少 critic 依赖、简化训练流程、提升对齐稳定性
  - 面试时不一定背细节，但要能讲清楚设计动机和权衡

# 工程实践：决定你能否落地
社招更看重你是否真正训练过模型、处理过不稳定训练、做过大规模实验。
##  分布式训练
需要掌握：
- 数据并行
  - 多卡训练 experience buffer
  - 梯度同步
  - batch size 扩大后的学习率调整
- 模型并行
  - 大模型参数过大，单卡放不下
  - tensor parallel、pipeline parallel
  - ZeRO 类优化
- Actor / Critic / Reference Model 分离
  - Actor：采样和策略更新
  - Critic：估计 value
  - Reference Model：计算 KL，防止策略偏移
  - Reward Model：提供奖励信号
- Rollout 和 Training 分离
  - rollout 负责生成轨迹
  - training 负责更新模型
  - 需要处理数据异步、版本不一致、experience staleness
  
##  训练稳定性
这是 RL 面试最核心的工程能力。
常见问题：

| 问题 | 可能原因 | 解决方向 |
|---|---|---|
| reward 暴涨但质量下降 | reward hacking | 加 KL、改 reward、加 verifier |
| KL 散度爆炸 | 策略偏离 reference 太远 | 降低学习率、加强 KL penalty |
| value loss 震荡 | advantage 尺度异常 | 归一化、clip value loss |
| 策略熵下降过快 | 过早收敛 | 加 entropy bonus |
| 梯度爆炸 | reward 尺度大、模型大 | gradient clipping、weight decay |
| 训练不收敛 | 数据分布漂移 | 更新 reference model、重采样 |

## Reward 建模
Reward Model 是大模型 RL 的关键。
需要掌握：
- 偏好数据构造
  - chosen / rejected
  - 多等级评分
  - 规则奖励 + 模型奖励
  - AI judge / verifier
- Reward Model 训练
  - 排序损失
  - 防止 reward scale 漂移
  - 防止过拟合 chosen 样本
  - 检查 reward hacking
- Reward 设计
  - 规则奖励：格式、安全、工具调用成功
  - 模型奖励：偏好、质量、对齐
  - 环境奖励：代码执行、任务完成、agent 反馈
  - 复合奖励：加权融合，但要注意尺度问题
  
## 数据管理
RL 数据管理比监督学习复杂很多。
需要关注：
  - prompt 分布
  - 采样温度
  - response 长度分布
  - chosen/rejected 质量
  - 重复样本
  - 低质量偏好标注
  - 数据污染
  - 离线数据与在线数据混合
  - 数据版本管理
  
## 调试工具
面试中如果能讲清楚调试方法，会非常加分。
常用手段：
  - torch.allclose 检查数值一致性
  - 对比单步 loss、gradient norm、logprob
  - 检查 advantage 均值和方差
  - 检查 KL 散度变化
  - 检查 reward 分布
  - 检查 policy entropy
  - 用小 batch、小模型复现 bug
  - 对比 CPU/GPU、float16/float32 结果
  - 检查 DataLoader、tokenizer、padding、mask

# 系统设计：考察综合能力
社招高级岗位通常会问系统设计，例如：
  - 如何设计一个大模型 RLHF 训练系统？
  - 如何设计一个 Agentic RL 训练链路？
  - 如何设计 Reward System？
  - 如何评估一个对齐模型？
  - 如何把 PPO 训练扩展到千卡规模？
  
##  大模型 RLHF 系统设计
可以按这个结构回答：

In [ ]:
SFT Model
   ↓
Prompt Sampler
   ↓
Actor Model  → 生成 response
   ↓
Reference Model → 计算 KL
   ↓
Reward Model → 计算 reward
   ↓
Critic Model → 估计 value
   ↓
PPO / DPO / GRPO Trainer
   ↓
Evaluation / Human Eval / Auto Judge
   ↓
Model Registry / Versioning

需要说明：
- Actor 负责采样
- Reference Model 负责约束策略偏移
- Reward Model 提供奖励信号
- Critic 提供价值估计
- Trainer 负责策略更新
- Evaluation 负责判断真实效果

## Agentic RL 系统设计
Agentic RL 比普通 RLHF 更复杂，因为涉及环境交互。
系统组成：
- Agent
  - 大模型作为决策器
  - 输出 action：工具调用、代码、搜索、推理步骤
- Environment
  - 代码执行环境
  - 浏览器环境
  - API 环境
  - 文件系统
  - 任务状态管理
- Reward System
  - 任务完成奖励
  - 工具调用成功奖励
  - 安全性惩罚
  - 步骤效率惩罚
  - 最终结果验证奖励
- Trajectory Manager
  - 记录完整交互轨迹
  - 处理长 horizon
  - 处理失败重试
  - 处理环境异步
- Training Loop
  - rollout 生成轨迹
  - 计算 reward
  - 计算 advantage
  - 更新 policy
  - 可选更新 critic / reward model

## Reward System 设计
Reward System 通常不是单一模型，而是多信号融合。
可以设计成：

In [ ]:
Rule Reward
   +
Model Reward
   +
Environment Reward
   +
Verifier Reward
   -
Safety Penalty
   →
Final Reward

需要重点说明：
- 不同 reward 的尺度归一化
- 避免某个 reward 主导训练
- 防止 reward hacking
- 加入 verifier 提高可信度
- 对安全类行为设置强惩罚
- 定期评估 reward 与真实人类偏好是否一致

## 评估体系
评估不能只看 reward，否则容易被 reward hacking 骗。
评估维度：
- 任务成功率
- 人类偏好胜率
- 安全性
- 格式遵循
- 工具调用成功率
- 推理质量
- 代码执行正确性
- 长任务完成率
- KL 散度
- reward 与人工评估的一致性

# 高频面试题清单
##  算法题
- 推导策略梯度公式
- 解释 PPO clip 的作用
- 解释 GAE 如何平衡偏差和方差
- 解释 DQN 为什么需要 target network
- 解释 Actor-Critic 中 Critic 为什么常用 MSE
- 解释 DPO 和 RLHF 的区别
- 解释 GRPO 相对 PPO 的改进动机
- 解释 KL 约束为什么重要
- 解释 reward hacking 及解决方法
- 解释 on-policy 和 off-policy 的区别
##  工程题
- PPO 训练不稳定怎么办？
- KL 散度爆炸怎么处理？
- Reward Model 过拟合怎么办？
- 大模型 RLHF 如何做分布式训练？
- Rollout 和 training 异步会带来什么问题？
- 如何判断 reward 是否可靠？
- 如何调试策略梯度方法？
- 如何设计 preference data 的清洗流程？
- 如何避免 chosen/rejected 数据污染？
- 如何评估一个对齐模型是否真的变好？
##  系统设计题
- 设计一个 RLHF 训练系统
- 设计一个 Agentic RL 训练系统
- 设计一个 Reward System
- 设计一个大模型对齐评估体系
- 设计一个支持 PPO/DPO/GRPO 的统一训练框架
- 如何把 RL 训练扩展到千卡规模？
- 如何设计一个安全的 coding agent？
- 如何设计一个长任务 agent 的 reward？

# 面试表达模板
回答算法题时，可以按这个顺序讲：

In [ ]:
1. 问题定义
2. 核心公式
3. 直觉解释
4. 优缺点
5. 工程中的坑
6. 改进方案

例如回答 PPO：

In [ ]:
PPO 是一种策略梯度方法，核心目标是在保证策略更新稳定的前提下提升期望回报。

它通过 ratio 衡量新旧策略差异，并用 clip 限制更新幅度，防止一步更新过大导致训练崩溃。

相比 vanilla policy gradient，PPO 更稳定；相比 TRPO，它实现更简单，不需要二阶优化。

工程上常见的问题是 KL 散度爆炸、reward hacking、value loss 不稳定。

通常可以通过 KL penalty、advantage normalization、gradient clipping、entropy bonus 来缓解。

# 项目经历准备模板
每个项目建议准备成 5 段：

In [ ]:
1. 背景：为什么要做这个项目
2. 目标：优化什么指标
3. 方法：用了什么 RL 算法、reward、训练策略
4. 难点：训练不稳定、reward hacking、数据质量、分布式训练
5. 结果：指标提升、线上效果、失败经验

例如：

In [ ]:
背景：原始模型在复杂指令遵循任务上表现不稳定。

目标：提升模型对格式、工具调用、安全约束的遵循能力。

方法：基于 PPO 进行后训练，使用 rule reward + model reward + verifier reward，并加入 KL 约束。

难点：训练初期出现 reward hacking，模型通过固定模板获得高 reward，但实际质量下降。

解决：引入 verifier 和格式惩罚，调整 reward 权重，降低 learning rate，并增加 KL penalty。

结果：人工评估胜率提升，reward 与人工打分一致性提高，线上 badcase 下降。

# 面试前一周冲刺清单
## 算法层
  - 能手推策略梯度
  - 能讲清 PPO clip 和 GAE
  - 能对比 PPO / DPO / GRPO
  - 能解释 RLHF 完整链路
  - 能说明 reward model 训练目标
  - 能解释 KL 约束的作用
## 工程层
  - 能讲分布式 RL 训练架构
  - 能讲 PPO 训练失败排查流程
  - 能讲 reward hacking 检测方法
  - 能讲 preference data 清洗流程
  - 能讲 rollout / training 异步问题
  - 能讲大模型训练中的显存优化
## 系统层
  - 能设计 RLHF 系统
  - 能设计 Agentic RL 系统
  - 能设计 Reward System
  - 能设计评估体系
  - 能讲清楚扩展性、稳定性、可观测性
  - 能讲清楚哪些模块最容易出问题

# 最终建议
强化学习社招面试的核心不是背公式，而是证明你具备三类能力：
## 算法理解能力
  - 知道为什么用 PPO、DPO、GRPO
  - 知道它们的偏差、方差、稳定性差异
## 工程落地能力
  - 能处理训练不稳定
  - 能处理 reward 建模问题
  - 能处理大规模分布式训练
## 系统设计能力
  - 能设计完整后训练链路
  - 能设计 Reward System
  - 能设计评估体系
  - 能识别系统瓶颈和风险
  
面试时尽量做到：

In [ ]:
公式能写出来；
直觉能讲清楚；
坑能举出来；
方案能对比；
系统能画出来。